# AI-assisted workflow

This notebook repeats the workflow of this week’s lesson, but this time an
AI assistant helps us write the code. The division of labour is simple:

- **You decide the GIS part**: which data, which coordinate reference system,
  which spatial predicate, which classes. You are the GIS expert, and these
  decisions go *into* your prompt.
- **The AI assistant helps with the Python part**: boilerplate, syntax you
  don’t remember, restructuring code, explaining error messages.

The responses shown below are *examples*. If you run the same prompts yourself,
you will get different code, and that is fine. What matters is that you can
read the code, run it, and check the result.

This week, we design the maps (what to show, which classes, which colours),
and use an AI assistant for the *matplotlib* and *folium* code to put them on
the screen.

> **Habit of the week: iterate**
>
> The first response is rarely the final one. Look at the result, decide what
> you want to change, and ask for exactly that in a follow-up prompt. Small,
> specific requests work much better than asking for ‘a better map’, and you
> stay in control of the cartographic decisions.

## Your decisions

- Compare travel times to the railway station by **public transport** (`pt_r_t`) and by **car** (`car_r_t`), in two maps next to each other.
- Cells with no data (`-1`) are excluded.
- **The same classes for both maps**, so they can be compared: 0–10, 10–20, 20–30, 30–40, 40–50, 50–60 and more than 60 minutes.
- Colour map: `RdYlBu` (red: close, blue: far).
- Interactive map: the address points, with the address shown when hovering over a point.

In [ ]:
import pathlib
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"

In [ ]:
import geopandas

accessibility_grid = geopandas.read_file(
    DATA_DIRECTORY
    / "helsinki_region_travel_times_to_railway_station"
    / "helsinki_region_travel_times_to_railway_station.gpkg"
)
accessibility_grid = accessibility_grid.loc[
    (accessibility_grid["pt_r_t"] >= 0)
    & (accessibility_grid["car_r_t"] >= 0)
]

## Step 1: a first version

**Prompt:**

```text
Using geopandas and matplotlib, plot two maps next to each other from the
GeoDataFrame `accessibility_grid`: the columns "pt_r_t" (public transport) and
"car_r_t" (car), travel times in minutes. Use the class breaks 10, 20, 30, 40,
50, 60 for both maps, the colour map RdYlBu and no cell borders. Add a title to
each map.
```

**Example response** (yours may differ):

In [ ]:
import matplotlib.pyplot as plt

BREAKS = [10, 20, 30, 40, 50, 60]

figure, axes = plt.subplots(ncols=2, figsize=(14, 6))

for ax, column, title in zip(axes, ["pt_r_t", "car_r_t"], ["Public transport", "Car"]):
    accessibility_grid.plot(
        ax=ax,
        column=column,
        scheme="UserDefined",
        classification_kwds={"bins": BREAKS},
        cmap="RdYlBu",
        linewidth=0,
        legend=True
    )
    ax.set_title(title)

**Look at the result**: the code works and uses our classes, but the figure
is not finished yet: there are two identical legends covering parts of the
maps, and the axes show coordinate values that nobody needs on this map. Let’s
ask for exactly these changes.

## Step 2: refine

**Prompt:**

```text
Change the figure: one shared legend below both maps (instead of one legend per
map), with the labels "0-10", "10-20", ..., "> 60" and the title "Travel time
(min)". Remove the axes (ticks and frames). Add an overall title "Travel time to
Helsinki railway station" and the source note "Data: Helsinki Region Travel Time
Matrix" in the bottom right corner.
```

**Example response** (yours may differ):

In [ ]:
import matplotlib
import mapclassify

LABELS = ["0-10", "10-20", "20-30", "30-40", "40-50", "50-60", "> 60"]
colour_map = matplotlib.colormaps["RdYlBu"]
colours = [colour_map(i / (len(LABELS) - 1)) for i in range(len(LABELS))]

figure, axes = plt.subplots(ncols=2, figsize=(14, 6))

for ax, column, title in zip(axes, ["pt_r_t", "car_r_t"], ["Public transport", "Car"]):
    # classify explicitly, so both maps use the same class numbers and colours
    classes = mapclassify.UserDefined(accessibility_grid[column], bins=BREAKS).yb
    accessibility_grid.assign(travel_time_class=classes).plot(
        ax=ax,
        column="travel_time_class",
        cmap="RdYlBu",
        vmin=0,
        vmax=len(LABELS) - 1,
        linewidth=0
    )
    ax.set_title(title)
    ax.set_axis_off()

legend_handles = [matplotlib.patches.Patch(color=colour) for colour in colours]
figure.legend(
    legend_handles,
    LABELS,
    title="Travel time (min)",
    loc="lower center",
    bbox_to_anchor=(0.5, 0.04),
    ncols=len(LABELS),
    frameon=False
)
figure.suptitle("Travel time to Helsinki railway station", fontsize=16)
figure.text(0.98, 0.01, "Data: Helsinki Region Travel Time Matrix", ha="right", fontsize=8)
figure.subplots_adjust(top=0.9, bottom=0.15, wspace=0.05)

**Before you trust it**, read the code: to draw one shared legend, the
response no longer uses `scheme=`, but classifies the data itself (class
numbers 0–6), fixes the colour range with `vmin` and `vmax`, and builds the
legend from coloured patches. This only works if the legend colours are
computed *in exactly the same way* as the map colours. That is worth a check:
do the class counts make sense, and does each legend colour match the colour
the map uses for that class?

In [ ]:
import numpy

for column in ["pt_r_t", "car_r_t"]:
    classes = mapclassify.UserDefined(accessibility_grid[column], bins=BREAKS).yb

    # the class of each cell must agree with the break values
    expected_classes = numpy.searchsorted(BREAKS, accessibility_grid[column], side="left")
    assert (classes == expected_classes).all()

    counts = numpy.bincount(classes, minlength=len(LABELS))
    print(column, dict(zip(LABELS, counts.tolist())))

# the colour of class i on the map is colour_map(normalised i), same as in the legend
normalise = matplotlib.colors.Normalize(vmin=0, vmax=len(LABELS) - 1)
assert all(colour_map(normalise(i)) == colours[i] for i in range(len(LABELS)))

## Step 3: an interactive map

**Prompt:**

```text
With folium, show the points of a GeoDataFrame `addresses` (column "address",
EPSG:4326) as small circle markers on a "cartodbpositron" base map. Show the
address in a tooltip when hovering over a point. Centre the map on the points.
```

**Example response** (yours may differ):

In [ ]:
import folium

addresses = geopandas.read_file(DATA_DIRECTORY / "addresses.gpkg", layer="addresses")

centre = addresses.to_crs("EPSG:3067").union_all().centroid
centre = geopandas.GeoSeries([centre], crs="EPSG:3067").to_crs("EPSG:4326").iloc[0]

interactive_map = folium.Map(
    location=(centre.y, centre.x),
    zoom_start=12,
    tiles="cartodbpositron"
)

folium.GeoJson(
    addresses,
    marker=folium.CircleMarker(radius=5, fill=True, fill_opacity=0.8),
    tooltip=folium.GeoJsonTooltip(fields=["address"], aliases=["Address:"])
).add_to(interactive_map)

interactive_map

**Before you trust it**, read it through:

- The centre point is computed in a projected CRS (EPSG:3067) and then
  transformed back to WGS84. (Computing a centroid directly on longitude and
  latitude values would produce a warning in geopandas.)
- folium expects `location` as **(latitude, longitude)**, i.e. *y* first, the
  opposite of shapely’s *(x, y)* order. The response uses `(centre.y,
  centre.x)`, which is correct. Whenever a map shows up in the wrong place,
  check the coordinate order first.

A quick check that the map is centred in Helsinki:

In [ ]:
assert 60.0 < centre.y < 60.4 and 24.5 < centre.x < 25.3, "The map centre is not in Helsinki"
print(f"Map centre: latitude {centre.y:.3f}, longitude {centre.x:.3f}")

## AI-use log

Whenever you use an AI assistant in this course (in exercises, only where a
task is marked **AI-LLM OK**, see the [course guidelines on AI
tools](https://autogis-site.readthedocs.io/en/latest/course-info/ai-tools.html)),
keep a short log of what you did. For this notebook, it could look like this:

| Step | What I asked | What I changed | How I checked it |
|------|--------------|----------------|------------------|
| 1 | Two maps next to each other with my classes and colour map | Asked for a follow-up (step 2) | Looked at the figure |
| 2 | One shared legend, no axes, title and source note | Nothing | Checked class counts and that legend colours match map colours |
| 3 | An interactive folium map with tooltips | Nothing | Checked the coordinate order and the map centre |